# P1: Define function to automate data cleaning process

In [233]:
# Import libraries
import pandas as pd
import numpy as np

# Display entire rows and columns for data inspection
pd.set_option('display.max_rows', None) 
pd.set_option('display.max_columns', None)


In [234]:
# Constants for pattern matching
REMOTE_MODES = ['remote', 'hybrid', 'on-site']
LEVELS = ['associate', 'graduate', 'senior', 'junior', 'entry-level', 'manager', 'lead']

def clean_data(file_path):
    """Main function to clean the job postings dataset."""
    df = pd.read_csv(file_path)

    # Data cleaning steps
    df = drop_duplicates(df)
    df = replace_blanks_with_nan(df)
    df = drop_unused_columns(df, ['Job URL'])
    df = capitalize_columns(df, ['Company', 'Title'])
    df = extract_working_mode_and_level(df)
    df = split_location(df)
    df = extract_and_clean_salary(df)
    df = handle_city_and_working_mode(df)
    df = extract_work_type_keywords(df)
    df = extract_pay_type_keywords(df)
    df['Date'] = pd.to_datetime(df['Date'])

    # Finalize the dataframe
    cleaned_df = df[['Date', 'Title', 'Company', 'State', 'City', 'Level', 'Work Mode', 'Average Salary',
                     'Pay Type','Employment Type','Description']]
    print(cleaned_df.isna().sum())  # Check for missing values
    return cleaned_df

def drop_duplicates(df):
    """Drop duplicate rows in the DataFrame."""
    return df.drop_duplicates()

def replace_blanks_with_nan(df):
    """Replace blank values with NaN."""
    return df.replace(' ', np.nan)

def drop_unused_columns(df, columns):
    """Drop specified unused columns from the DataFrame."""
    return df.drop(columns=columns)

def capitalize_columns(df, columns):
    """Capitalize text in specified columns."""
    for column in columns:
        df[column] = df[column].astype(str).apply(lambda x: ' '.join(word.capitalize() for word in x.split()))
    return df

def extract_working_mode_and_level(df):
    """Extract Work Mode and Level from the Title."""
    
    def find_remote_mode(title):
        title_lower = str(title).lower()
        for mode in REMOTE_MODES:
            if mode in title_lower:
                return mode.capitalize()
        return None
    
    def find_level(title):
        title_lower = str(title).lower()
        for level in LEVELS:
            if level in title_lower:
                return level.capitalize()
        return 'Mid-level'
    
    df['Work Mode'] = df['Title'].apply(find_remote_mode)
    df['Level'] = df['Title'].apply(find_level)
    
    return df

def split_location(df):
    """Split 'Location' into 'City' and 'Post_Code' without regex."""
    df[['City', 'Post_Code']] = df['Location'].str.split(',', n=1, expand=True)
    
    # Keep only digits for postcode
    df['Post_Code'] = df['Post_Code'].apply(lambda x: ''.join(c for c in str(x) if c.isdigit()) if pd.notnull(x) else np.nan)
    return df

def extract_and_clean_salary(df):
    """Extract and clean salary information."""
    df['Salary'] = df['Salary'].astype(str).apply(lambda x: 'Day Shift' if 'shift' in str(x).lower() else x)
    df['Salary1'] = df['Salary'].apply(lambda x: ''.join(c for c in str(x) if c.isdigit() or c in '.,-')).str.strip('-')
    
    # Fill missing salaries from description if available
    def extract_salary_from_description(desc):
        if pd.isna(desc):
            return None
        for word in desc.split():
            word_clean = word.replace('$', '').replace(',', '')
            try:
                return float(word_clean)
            except:
                continue
        return None
    
    df['Salary2'] = df['Description'].apply(extract_salary_from_description)
    df['Salary1'] = df['Salary1'].replace('', np.nan).fillna(df['Salary2'])
    df['Salary1'] = df['Salary1'].replace('1', np.nan).replace('2', np.nan).replace('3', np.nan)
    
    # Separate into Min and Max salary
    df[['Min_salary', 'Max_salary']] = df['Salary1'].str.split('-', n=1, expand=True)
    df['Max_salary'] = df['Max_salary'].fillna(df['Min_salary'])
    
    for col in ['Min_salary', 'Max_salary']:
        df[col] = df[col].apply(lambda x: float(str(x).replace('$', '').replace(',', '')) if pd.notnull(x) else np.nan)
    
    # Calculate average salary
    df['Average Salary'] = (df['Min_salary'] + df['Max_salary']) / 2

    mask = df['Salary'].str.contains('K', case=False, na=False)

    # Multiply Average Salary by 1000 only for those rows
    df.loc[mask, 'Average Salary'] *= 1000

    return df

# create new column salary pay type
def extract_work_type_keywords(df):
    """Create Work Type column based on salary text keywords."""
    def get_work_type(row):

        if pd.isna(row['Average Salary']):
            return row['Salary']
    df['Employment Type'] = df.apply(get_work_type, axis=1)
    return df

# create new column salary pay type
def extract_pay_type_keywords(df):
    """Create Work Type column based on salary text keywords."""
    salary_text = str(df['Salary']).lower()
    def get_pay_type(row):
        salary_text = str(row['Salary']).lower()
        # Look for keywords
        for keyword in ['year', 'month', 'week', 'day', 'hour']:
            if keyword in salary_text:
                return keyword
        return None  
    
    # Apply function row-wise
    df['Pay Type'] = df.apply(get_pay_type, axis=1)
    return df

def handle_city_and_working_mode(df):
    """Handle City and Work Mode for remote jobs."""
    df['City'] = np.where(df['State'] == 'Remote', 'Remote', df['City'])
    df['City'] = df['City'].apply(lambda x: x.split(' in ')[-1].strip() if isinstance(x, str) and 'in' in x else x)
    return df


In [235]:
# File path to be defined
file_path = 'indeed_kaggle.csv'

# apply function to the uncleaned data
cleaned_df = clean_data(file_path)

Date                   0
Title                  0
Company                0
State                  0
City                   2
Level                  0
Work Mode          28007
Average Salary     17806
Pay Type           17747
Employment Type    11378
Description         6429
dtype: int64


In [236]:
#inspect the data
cleaned_df[cleaned_df['Average Salary'].notna() & cleaned_df['Pay Type'].isna()]

,Date,Title,Company,State,City,Level,Work Mode,Average Salary,Pay Type,Employment Type,Description
4205,2024-06-23,"Scientist, Electronic Warfare Analyst",L3harris Technologies,Utah,Salt Lake City,Mid-level,None,980.0,None,None,Bachelor’s Degree and a minimum of 12 years of...


As `9x80` is not salary, so remove the salary and avg salary from this record

In [237]:
cleaned_df.loc[4205, 'Average Salary'] = np.nan
cleaned_df.loc[4205, 'Pay Type'] = np.nan

In [238]:
#Save cleaned dataset to CSV
cleaned_df.to_csv('completed_file.csv', index=False)

In [240]:
uncleaned_df = pd.read_csv(file_path)


,Title,Company,Location,Salary,Description,Job URL,Date,State


In [248]:
pd.set_option('display.max_colwidth', None)
uncleaned_df.loc[uncleaned_df['Company'].str.contains('3cloud', case=False, na=False)]['Description']


12922    Minimum of 5 years of experience with data science, or machine learning work.\nWe are looking for a Data Science Architect who will be responsible for delivering…
14981                                Minimum of 5 years of experience with data science (or machine learning work), Azure technologies, and previous Consulting experience.
14991    Minimum of 5 years of experience with data science, or machine learning work.\nWe are looking for a Data Science Architect who will be responsible for delivering…
16300         Are you looking for a role that motivates and challenges you? Are you ready for an opportunity for growth? Do you want to work on teams where people roll up…
16962         Are you looking for a role that motivates and challenges you? Are you ready for an opportunity for growth? Do you want to work on teams where people roll up…
17155    Minimum of 5 years of experience with data science, or machine learning work.\nWe are looking for a Data Science Architect who will